# Week 04 — Scatter & Bubble Charts: Gapminder
## Asia: changes over time and differences across countries

Completed exercises based on `week04/lecture04_exercise.ipynb` and the design principles in `lecture04_notes.ipynb`.

**Run in Kaggle:** Create a Python notebook, choose **File → Import Notebook**, upload this `.ipynb`, then run all cells. Use the standard Python environment; no accelerator, additional dataset upload, or internet access is needed. The data are included in Plotly through `px.data.gapminder()` as specified in the exercise.

**Design rules followed:** GDP per capita uses a logarithmic axis; Task 1 uses only two colours for year and one continent other than Africa; Task 2 highlights Asia and greys all other countries, with population mapped to bubble area and an explicit `size_max`. Both charts have insight titles and annotations.

**Data:** Historical Gapminder sample supplied by Plotly (1952–2007), attributed to the Gapminder Foundation. This is a historical teaching dataset, not a description of present-day conditions. GDP per capita is in the dataset's inflation-adjusted international dollars (PPP); life expectancy is in years and population is in people.

In [ ]:
import math
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown

# Embed the Plotly JavaScript so charts do not require a CDN connection.
pio.renderers.default = "notebook"
px.defaults.template = "plotly_white"

df = px.data.gapminder()
print(f"Loaded {len(df):,} rows; {df['country'].nunique()} countries.")
display(df.head())

### Explore and check the data
Each row describes one country in one year. We check completeness, unique country–year pairs and positive values before filtering. No observations are removed from the full 2007 sample in Task 2.

In [ ]:
df.info()
print("Years:", sorted(df["year"].unique().tolist()))
print("Continents:", sorted(df["continent"].unique().tolist()))
display(df.describe().round(1))
display(df.isna().sum().rename("Missing values").to_frame())

assert not df.duplicated(["country", "year"]).any()
assert df[["country", "continent", "year", "gdpPercap", "lifeExp", "pop"]].notna().all().all()
assert (df[["gdpPercap", "lifeExp", "pop"]] > 0).all().all()

## Task 1 — Scatter: change from 2002 to 2007 in Asia

GDP per capita is on the x-axis and life expectancy on the y-axis. Blue represents 2002 and orange represents 2007, using two colours from the colour-vision-friendly Okabe–Ito palette. Neutral arrows link the same country from 2002 to 2007: right means higher GDP per capita and up means longer life expectancy. Hover over a point to identify the country.

A dotted line shows the **unweighted median life expectancy across Asian countries in 2007**, providing a reference without adding another colour category.

In [ ]:
continent = "Asia"
years = [2002, 2007]
asia = df.loc[(df["continent"] == continent) & df["year"].isin(years)].copy()
asia["Year"] = asia["year"].astype(str)  # categorical colour, not a continuous scale

start = asia.loc[asia["year"] == 2002].set_index("country")
end = asia.loc[asia["year"] == 2007].set_index("country")
assert set(start.index) == set(end.index)
change = pd.DataFrame({
    "GDP per capita change (%)": (end["gdpPercap"] / start["gdpPercap"] - 1) * 100,
    "Life expectancy change (years)": end["lifeExp"] - start["lifeExp"],
}).sort_index()
improved_both = int((change > 0).all(axis=1).sum())
display(change.round(2))
print(f"Both measures increased in {improved_both} of {len(change)} Asian countries.")

In [ ]:
fig1 = px.scatter(
    asia, x="gdpPercap", y="lifeExp", color="Year",
    category_orders={"Year": ["2002", "2007"]},
    color_discrete_map={"2002": "#0072B2", "2007": "#D55E00"},
    hover_name="country", hover_data={"gdpPercap": ":,.0f", "lifeExp": ":.1f", "Year": True},
    log_x=True,
    labels={"gdpPercap": "GDP per capita (PPP international $, log scale)",
            "lifeExp": "Life expectancy (years)"},
    title=f"Asia: {improved_both} of {len(change)} countries gained income and longevity"
          "<br><sup>2002 → 2007 · arrows link the same country</sup>",
    height=650,
)
fig1.update_traces(marker=dict(size=9, opacity=0.9, line=dict(width=0.5, color="white")))

# Annotation positions on a Plotly logarithmic axis use log10 coordinates.
# Matching axref/ayref to xref/yref makes each arrow start at the actual 2002 point.
for country in start.index:
    old, new = start.loc[country], end.loc[country]
    fig1.add_annotation(
        x=math.log10(new["gdpPercap"]), y=new["lifeExp"],
        ax=math.log10(old["gdpPercap"]), ay=old["lifeExp"],
        xref="x", yref="y", axref="x", ayref="y",
        text="", showarrow=True, arrowhead=2, arrowsize=0.8,
        arrowwidth=1, arrowcolor="#737373", opacity=0.65,
        standoff=5, startstandoff=5,
    )

median_2007 = float(end["lifeExp"].median())
fig1.add_hline(y=median_2007, line_dash="dot", line_color="#999999",
              annotation_text=f"2007 country median: {median_2007:.1f} years",
              annotation_position="bottom right")

# Directly label one notable country; leave other labels available on hover.
china = end.loc["China"]
fig1.add_annotation(x=math.log10(china["gdpPercap"]), y=china["lifeExp"],
                    text="China, 2007", showarrow=True, ax=-60, ay=35,
                    arrowcolor="#555555", bgcolor="white", borderpad=3)
fig1.update_layout(font=dict(family="Arial", size=13),
                   margin=dict(t=115, b=70, l=75, r=35),
                   legend=dict(orientation="h", x=0, y=1.08, title="Year"))
fig1.update_xaxes(gridcolor="#EEEEEE")
fig1.update_yaxes(showgrid=False)
fig1.show()

In [ ]:
display(Markdown(
    f"**Interpretation.** Between 2002 and 2007, **{improved_both} of {len(change)}** "
    f"Asian countries increased both measures. The median country-level GDP per capita "
    f"change was **{change['GDP per capita change (%)'].median():.1f}%**, while the "
    f"median country-level life expectancy change was "
    f"**{change['Life expectancy change (years)'].median():.2f} years**. "
    "These are medians of individual country changes, not population-weighted measures. "
    "The arrows show paired observations; they do not establish that income growth caused longer lives."
))

## Task 2 — Bubble chart: Asia spans a wide life expectancy range

Use **every country in the 2007 dataset**. Asia is blue; all other continents are grey. Bubble **area** represents population, with `size_max=55` to limit overlap. The lowest- and highest-life-expectancy Asian countries are annotated to explain the highlighted group's diversity.

The dotted line is the unweighted median life expectancy across all countries in the 2007 sample. It is a country reference value, not the life expectancy of the world's population.

In [ ]:
world = df.loc[df["year"] == 2007].copy()
world["Group"] = world["continent"].map(lambda c: "Asia" if c == "Asia" else "Other continents")
# Draw larger bubbles first within each group, then highlight Asia on top.
world = world.sort_values("pop", ascending=False)
asia_2007 = world.loc[world["continent"] == "Asia"]
low = asia_2007.loc[asia_2007["lifeExp"].idxmin()]
high = asia_2007.loc[asia_2007["lifeExp"].idxmax()]
life_gap = float(high["lifeExp"] - low["lifeExp"])
world_median = float(world["lifeExp"].median())

fig2 = px.scatter(
    world, x="gdpPercap", y="lifeExp", size="pop", size_max=55,
    color="Group", category_orders={"Group": ["Other continents", "Asia"]},
    color_discrete_map={"Other continents": "#BDBDBD", "Asia": "#0072B2"},
    hover_name="country", custom_data=["continent", "pop"],
    log_x=True,
    labels={"gdpPercap": "GDP per capita (PPP international $, log scale)",
            "lifeExp": "Life expectancy (years)", "pop": "Population", "Group": ""},
    title=f"Asia's life expectancy gap spans {life_gap:.0f} years in 2007"
          "<br><sup>All countries in the sample · bubble area represents population</sup>",
    height=680,
)
fig2.update_traces(
    marker=dict(line=dict(color="white", width=0.6)),
    hovertemplate="<b>%{hovertext}</b><br>Continent: %{customdata[0]}"
                  "<br>GDP per capita: $%{x:,.0f}"
                  "<br>Life expectancy: %{y:.1f} years"
                  "<br>Population: %{customdata[1]:,.0f}<extra></extra>",
)
fig2.update_traces(selector=dict(name="Other continents"), marker_opacity=0.5)
fig2.update_traces(selector=dict(name="Asia"), marker_opacity=0.85)
fig2.add_hline(y=world_median, line_dash="dot", line_color="#999999",
              annotation_text=f"World sample country median: {world_median:.1f} years",
              annotation_position="bottom right")

for row, ax, ay in [(low, 90, -60), (high, -85, 50)]:
    fig2.add_annotation(
        x=math.log10(row["gdpPercap"]), y=row["lifeExp"],
        text=f"<b>{row['country']}</b><br>{row['lifeExp']:.1f} years",
        showarrow=True, arrowhead=2, ax=ax, ay=ay,
        arrowcolor="#0072B2", font=dict(color="#0072B2", size=12),
        bgcolor="white", bordercolor="#0072B2", borderwidth=1, borderpad=5,
    )
fig2.update_layout(font=dict(family="Arial", size=13),
                   margin=dict(t=115, b=70, l=75, r=35),
                   legend=dict(orientation="h", x=0, y=1.08, itemsizing="constant"))
fig2.update_xaxes(gridcolor="#EEEEEE")
fig2.update_yaxes(showgrid=False, range=[35, 90])
fig2.show()

In [ ]:
population_share = 100 * asia_2007["pop"].sum() / world["pop"].sum()
display(Markdown(
    f"**Interpretation.** Asia contains both **{low['country']} ({low['lifeExp']:.1f} years)** "
    f"and **{high['country']} ({high['lifeExp']:.1f} years)**, a life expectancy difference "
    f"of **{life_gap:.1f} years**. A single continent label therefore conceals considerable "
    f"variation. Asia accounts for **{population_share:.1f}% of the population represented "
    "in this 2007 sample**, making its large bubbles especially prominent. "
    "The chart shows an association between income and life expectancy, not a causal effect."
))
display(asia_2007[["country", "gdpPercap", "lifeExp", "pop"]]
        .sort_values("lifeExp").round({"gdpPercap": 0, "lifeExp": 1}))

### Verify the exercise requirements
These checks verify the filtered years, one-continent restriction, full 2007 coverage, categorical colour mapping and population-based bubble sizing. The bubble chart retains one common area scale across its two colour groups.

In [ ]:
assert set(asia["year"]) == {2002, 2007}
assert set(asia["continent"]) == {"Asia"}
assert len(asia) == 2 * len(change)
assert fig1.layout.xaxis.type == fig2.layout.xaxis.type == "log"
assert {trace.name for trace in fig1.data} == {"2002", "2007"}
assert len({trace.marker.color for trace in fig1.data}) == 2
assert len(world) == len(df.loc[df["year"] == 2007]) == 142
assert world["country"].nunique() == 142
assert sum(len(trace.x) for trace in fig2.data) == 142
assert {trace.name: trace.marker.color for trace in fig2.data} == {
    "Other continents": "#BDBDBD", "Asia": "#0072B2"
}
assert all(trace.marker.sizemode == "area" for trace in fig2.data)
assert len({trace.marker.sizeref for trace in fig2.data}) == 1
assert math.isclose(fig2.data[0].marker.sizeref, world["pop"].max() / 55**2)
print("All exercise requirement checks passed.")

### Conclusions and limitations
- The paired-year scatter distinguishes changes within countries from differences between countries.
- The bubble chart shows that Asia contains very different income and life expectancy outcomes; bubble area supplies population context without adding a second colour variable.
- A log GDP axis makes proportional income differences readable. Country-level associations do not establish causation or describe inequality within each country.
- The sample includes 142 countries in 2007, not every country or territory. Country medians are unweighted; sample population shares are not presented as exact world population shares.

**Sources:** the supplied Week 04 exercise and lecture notes; `plotly.express.data.gapminder()` (Gapminder Foundation dataset).